In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 300)

BASE_URL = "https://te4.org/characters-vault"
SITE_URL = "https://te4.org"

MOTS_CLES_INTERDITS = {
    "god",
    "godmode",
    "cheat",
    "experience",
    "tougher escorts",
    "generous",
    "starting prodigy",
    "overpowered",
    "homosuperior",
    "superhuman",
    "expanded shop",
    "no more rare monsters",
    "softcore death",
    "hulk",
    "no prodigy requirement",
    "select your escorts",
    "exponential leveling"
}

PARAMS_BASE = {
    "tag_name": "",
    "tag_level_min": "",
    "tag_level_max": "",
    "tag_winner": "winner",
    "tag_permadeath[]": "66",     # 72=aventure, 66=roguelike
    "tag_difficulty[]": "36",    # 6=normal, 26=nightmare, 36=insane, 227=madness
    "tag_class[]": "23313",          # Berserker
    "tag_campaign[]": "2",        # Age of Ascendancy
}

class_name = "Doombringer"
TALENT_LANGUE_ANGLAISE = "rush"

def get_html(url, params=None, timeout=30):
    reponse = requests.get(url, params=params, timeout=timeout)
    reponse.raise_for_status()
    return reponse.text


def contient_mot_interdit(textes):
    texte = " ".join(textes).lower()

    for mot in MOTS_CLES_INTERDITS:
        if mot.lower() in texte:
            return True, mot

    return False, None


def supprimer_tooltips(bloc):
    for tooltip in bloc.find_all(class_="qtip-tooltip"):
        tooltip.decompose()


def extraire_artefacts_jaunes(html):
    soup = BeautifulSoup(html, "html.parser")

    for titre in soup.find_all("h4"):
        if titre.get_text(strip=True) != "Equipment":
            continue

        tableau = titre.find_next("table")
        if tableau is None:
            return []

        supprimer_tooltips(tableau)

        artefacts = []

        for span in tableau.select('span[style*="#FFD700"]'):
            nom = span.get_text(" ", strip=True)
            if nom:
                artefacts.append(nom)

        return artefacts

    return []


def extraire_page(html):
    soup = BeautifulSoup(html, "html.parser")
    lignes = soup.select("table.sticky-enabled tbody tr")

    donnees = []

    for ligne in lignes:
        cellules = ligne.find_all("td")

        if len(cellules) < 8:
            continue

        lien = cellules[1].find("a")
        url = lien.get("href") if lien else None

        if url and url.startswith("/"):
            url = SITE_URL + url

        donnees.append({
            "user": cellules[0].get_text(strip=True),
            "name": cellules[1].get_text(strip=True),
            "character_url": url,
            "class": cellules[2].get_text(strip=True),
            "difficulty": cellules[3].get_text(strip=True),
            "permadeath": cellules[4].get_text(strip=True),
            "campaign": cellules[5].get_text(strip=True),
            "winner": cellules[6].get_text(strip=True),
            "last_updated": cellules[7].get_text(strip=True),
        })

    return pd.DataFrame(donnees)


def scraper_pages(nb_pages=50):
    tableaux = []

    for page in range(nb_pages):
        html = get_html(BASE_URL, params=PARAMS_BASE | {"page": page})
        df_page = extraire_page(html)

        if df_page.empty:
            print(f"Stop page {page}")
            break

        print(f"Page {page}: {len(df_page)} lignes")
        tableaux.append(df_page)

    return pd.concat(tableaux, ignore_index=True) if tableaux else pd.DataFrame()


def extraire_addons(html):
    soup = BeautifulSoup(html, "html.parser")

    for cellule in soup.find_all("td"):
        if cellule.get_text(strip=True) != "Addons":
            continue

        cellule_addons = cellule.find_next_sibling("td")
        if cellule_addons is None:
            return []

        supprimer_tooltips(cellule_addons)

        addons = []

        for bloc in cellule_addons.find_all("div", class_="addon"):
            nom = bloc.get_text(" ", strip=True)
            if nom:
                addons.append(nom)

        return sorted(set(addons))

    return []


def extraire_prodigies(html):
    soup = BeautifulSoup(html, "html.parser")

    for titre in soup.find_all("h4"):
        if titre.get_text(strip=True) != "Prodigies":
            continue

        tableau = titre.find_next("table", class_="talents")
        if tableau is None:
            return []

        prodigies = []

        for ligne in tableau.find_all("tr"):
            cellule = ligne.find("td")
            if cellule is None:
                continue

            supprimer_tooltips(cellule)

            nom = cellule.get_text(" ", strip=True)
            if nom:
                prodigies.append(nom)

        return prodigies

    return []

def calculer_points_moyens_talents(df_details):
    lignes = []

    for _, ligne in df_details.loc[~df_details["ignore"]].iterrows():
        for talent in ligne["talents"]:
            lignes.append({
                "index_df": ligne["index_df"],
                **talent,
            })

    colonnes = [
        "index_df",
        "type_talent",
        "arbre",
        "multiplicateur_arbre",
        "talent",
        "points",
        "points_max",
    ]

    df_talents = pd.DataFrame(lignes, columns=colonnes)

    if df_talents.empty:
        return df_talents, pd.DataFrame(columns=[
            "type_talent",
            "arbre",
            "talent",
            "points_moyens",
            "points_medians",
            "nb_personnages",
            "points_max",
        ])

    df_moyennes = (
        df_talents
        .groupby(["type_talent", "arbre", "talent"], dropna=False)
        .agg(
            points_moyens=("points", "mean"),
            points_medians=("points", "median"),
            points_mode=("points", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
            nb_personnages=("points", "count"),
            points_max=("points_max", "max"),
        )
        .sort_values(["type_talent", "points_moyens"], ascending=[True, False])
        .reset_index()
    )

    return df_talents, df_moyennes


def analyser_personnage(url):
    html = get_html(url, timeout=10)

    addons = extraire_addons(html)
    ignore, mot = contient_mot_interdit(addons)

    resultat = {
        "url": url,
        "ignore": ignore,
        "mot_interdit": mot,
        "addons": addons,
        "prodigies": [],
        "artefacts_jaunes": [],
        "talents": [],
    }

    if ignore:
        return resultat

    talents = extraire_class_et_generic_talents(html)

    # filtre langue : on garde seulement si "Rush" existe
    noms_talents = {t["talent"].lower() for t in talents}
    if TALENT_LANGUE_ANGLAISE not in noms_talents:
        resultat["ignore"] = True
        resultat["mot_interdit"] = "langue_non_anglaise"
        return resultat

    resultat["talents"] = talents
    resultat["prodigies"] = extraire_prodigies(html)
    resultat["artefacts_jaunes"] = extraire_artefacts_jaunes(html)

    return resultat


def extraire_talents(html, titre_talents):
    soup = BeautifulSoup(html, "html.parser")
    talents = []

    titre = soup.find("h4", string=lambda texte: texte and texte.strip() == titre_talents)
    if titre is None:
        return talents

    tableau = titre.find_next("table", class_="talents")
    if tableau is None:
        return talents

    arbre_courant = None
    multiplicateur_arbre = None

    for ligne in tableau.find_all("tr"):
        cellules = ligne.find_all("td")

        if len(cellules) == 2 and cellules[0].find("strong"):
            arbre_courant = cellules[0].get_text(" ", strip=True)
            multiplicateur_arbre = cellules[1].get_text(strip=True)
            continue

        if len(cellules) < 2:
            continue

        valeur = cellules[-1].get_text(strip=True)
        match = re.search(r"(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)", valeur)

        if not match:
            continue

        cellule_nom = cellules[0]
        supprimer_tooltips(cellule_nom)

        nom = cellule_nom.get_text(" ", strip=True)

        talents.append({
            "type_talent": titre_talents,
            "arbre": arbre_courant,
            "multiplicateur_arbre": multiplicateur_arbre,
            "talent": nom,
            "points": float(match.group(1)),
            "points_max": float(match.group(2)),
        })

    return talents


def extraire_class_et_generic_talents(html):
    return (
        extraire_talents(html, "Class Talents")
        + extraire_talents(html, "Generic Talents")
    )


def compter_prodigies_et_artefacts(df):
    compteur_prodigies = {}
    compteur_artefacts = {}
    details = []

    for i, url in enumerate(df["character_url"]):
        try:
            resultat = analyser_personnage(url)
            resultat["index_df"] = i
            details.append(resultat)

            if resultat["ignore"]:
                print(f"[IGNORE] #{i} | mot='{resultat['mot_interdit']}'")
                continue

            for prodigy in set(resultat["prodigies"]):
                compteur_prodigies[prodigy] = compteur_prodigies.get(prodigy, 0) + 1

            for artefact in set(resultat["artefacts_jaunes"]):
                compteur_artefacts[artefact] = compteur_artefacts.get(artefact, 0) + 1

            if (i + 1) % 20 == 0:
                print(f"{i + 1} persos traités")

        except Exception as e:
            print(f"Erreur #{i} | {url} | {type(e).__name__}: {e}")

    df_prodigies = (
        pd.DataFrame(compteur_prodigies.items(), columns=["prodigy", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    df_artefacts = (
        pd.DataFrame(compteur_artefacts.items(), columns=["artefact", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    return df_prodigies, df_artefacts, pd.DataFrame(details)


df = scraper_pages(50)

df["race"] = df["name"].str.extract(
    rf"level\s+\d+\s+(\w+)\s+{class_name}",
    expand=False)

compte_par_race = (
    df["race"]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

df_prodigies, df_artefacts, df_details = compter_prodigies_et_artefacts(df)

index_pris_en_compte = df_details.loc[~df_details["ignore"], "index_df"]
df_filtre = df.loc[index_pris_en_compte].reset_index(drop=True)

compte_par_race_filtre = (
    df_filtre["race"]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

df_talents, df_talents_moyens = calculer_points_moyens_talents(df_details)

display(df_filtre.head(15))
display(df_prodigies)
display(df_artefacts.head(100))

df_talents_top = (
    df_talents_moyens
    .loc[df_talents_moyens["nb_personnages"] >= 10]
    .sort_values("points_moyens", ascending=False)
    .reset_index(drop=True)
)

display(df_talents_top)

display(compte_par_race_filtre)

Page 0: 25 lignes
Page 1: 25 lignes
Page 2: 25 lignes
Page 3: 25 lignes
Page 4: 25 lignes
Page 5: 25 lignes
Page 6: 25 lignes
Page 7: 25 lignes
Page 8: 13 lignes
Stop page 9
[IGNORE] #0 | mot='select your escorts'
[IGNORE] #1 | mot='select your escorts'
[IGNORE] #2 | mot='select your escorts'
[IGNORE] #8 | mot='langue_non_anglaise'
[IGNORE] #10 | mot='langue_non_anglaise'
[IGNORE] #11 | mot='langue_non_anglaise'
[IGNORE] #12 | mot='select your escorts'
[IGNORE] #13 | mot='tougher escorts'
[IGNORE] #14 | mot='langue_non_anglaise'
[IGNORE] #15 | mot='langue_non_anglaise'
[IGNORE] #17 | mot='langue_non_anglaise'
[IGNORE] #18 | mot='select your escorts'
[IGNORE] #19 | mot='langue_non_anglaise'
[IGNORE] #24 | mot='select your escorts'
[IGNORE] #26 | mot='select your escorts'
[IGNORE] #27 | mot='langue_non_anglaise'
[IGNORE] #29 | mot='langue_non_anglaise'
[IGNORE] #31 | mot='softcore death'
40 persos traités
[IGNORE] #41 | mot='langue_non_anglaise'
[IGNORE] #45 | mot='select your escorts'
[

,user,name,character_url,class,difficulty,permadeath,campaign,winner,last_updated,race
0,tsvit,Mebo the level 50 Dwarf Doombringer,https://te4.org/characters/399137/tome/9e208d8a-ec81-424e-b4b3-06bf021dbe54,doombringer,insane,roguelike,maj'eyal,yes,2 weeks 2 days ago,Dwarf
1,Siewiorr,Doul the level 50 Ghoul Doombringer,https://te4.org/characters/285927/tome/be257945-12dd-4eb0-8c27-a5f78f8577ad,doombringer,insane,roguelike,maj'eyal,yes,2 weeks 2 days ago,Ghoul
2,prodo,Dr.Insano the level 50 Skeleton Doombringer,https://te4.org/characters/398635/tome/8f09200c-68d0-4ce6-ba95-f347c28acfc2,doombringer,insane,roguelike,maj'eyal,yes,3 weeks 6 days ago,Skeleton
3,wtsai89,little destroyer the level 50 Halfling Doombringer,https://te4.org/characters/128024/tome/c0bcec53-12e0-4c02-8853-3c5a6332f25a,doombringer,insane,roguelike,maj'eyal,yes,3 weeks 6 days ago,Halfling
4,KannKarate,Ironclad the level 50 Ogre Doombringer,https://te4.org/characters/218167/tome/9737a6bd-5ac9-4e7c-aeb4-e77e6f67cd8d,doombringer,insane,roguelike,maj'eyal,yes,5 weeks 11 hours ago,Ogre
5,Gorbleezi,Riala the level 50 Shalore Doombringer,https://te4.org/characters/224193/tome/74339c16-ec86-4e3c-ae58-0e68518e1f99,doombringer,insane,roguelike,maj'eyal,yes,9 weeks 5 days ago,Shalore
6,malirain,Kabas the level 50 Ogre Doombringer,https://te4.org/characters/255494/tome/bcc6f123-ba04-478a-aa30-b074cff71f33,doombringer,insane,roguelike,maj'eyal,yes,19 weeks 5 days ago,Ogre
7,Siewiorr,Skelly of doom the level 50 Skeleton Doombringer,https://te4.org/characters/285927/tome/7625977d-3761-4ca9-a8bf-e5019d9492fe,doombringer,insane,roguelike,maj'eyal,yes,21 weeks 1 day ago,Skeleton
8,Daricus,Flynn Taggart the level 50 Ogre Doombringer,https://te4.org/characters/266101/tome/08a8ada8-191f-49a6-bebf-a96288c368b6,doombringer,insane,roguelike,maj'eyal,yes,23 weeks 5 days ago,Ogre
9,firmament,Euijoo the level 50 Ogre Doombringer,https://te4.org/characters/261241/tome/07e16a73-5468-4862-b220-d9fcb89e414e,doombringer,insane,roguelike,maj'eyal,yes,24 weeks 4 days ago,Ogre


,prodigy,count
0,Flexible Combat,70
1,Arcane Might,59
2,Ethereal Form,54
3,I Can Carry The World!,45
4,Adept,12
5,Cauterize,12
6,Corrupted Shell,7
7,Pain Enhancement System,5
8,Irresistible Sun,4
9,Spine of the World,3


,artefact,count
0,"Dakhtun's Gauntlets (0 def, 6 armour)",68
1,"Unbreakable Greaves (8 def, 20 armour)",38
2,"The Black Plate (25 def, 35 armour)",35
3,Limmir's Amulet of the Moon,30
4,Wheel of Fate,26
5,"Dethblyd (182% power, 18 apr)",22
6,"Dethblyd (70-112 power, 18 apr)",20
7,"Boots of the Hunter (2 def, 12 armour)",19
8,"Will of Ul'Gruth (0 def, 15 armour)",18
9,"Aetherwalk (6 def, 0 armour)",17


,type_talent,arbre,talent,points_moyens,points_medians,points_mode,nb_personnages,points_max
0,Class Talents,Corruption / Wrath,Destroyer,4.869565,5.0,5.0,138,5.0
1,Class Talents,Corruption / Brutality,Draining Assault,4.571429,5.0,5.0,140,5.0
2,Class Talents,Corruption / Torture,Abduction,4.371429,5.0,5.0,140,5.0
3,Class Talents,Corruption / Brutality,Reckless Strike,4.285714,5.0,5.0,140,5.0
4,Class Talents,Corruption / Wrath,Obliterating Smash,4.115942,5.0,5.0,138,5.0
5,Class Talents,Technique / Combat techniques,Blinding Speed,4.050000,5.0,5.0,140,5.0
6,Class Talents,Corruption / Shadowflame,Flame of Urh'Rok,4.035714,5.0,5.0,140,5.0
7,Class Talents,Corruption / Brutality,Fiery Grasp,3.307143,4.0,4.0,140,5.0
8,Class Talents,Corruption / Fearfire,Cauterize Spirit,3.148760,3.0,3.0,121,5.0
9,Class Talents,Corruption / Torture,Incinerating Blows,2.985714,4.0,1.0,140,5.0


,race,count
0,Ogre,35
1,Cornac,17
2,Halfling,16
3,Shalore,12
4,Skeleton,12
5,Dwarf,11
6,Drem,9
7,Higher,8
8,Ghoul,6
9,Doomelf,6


,type_talent,arbre,talent,points_moyens,points_medians,points_mode,nb_personnages,points_max
0,Generic Talents,Undead / Skeleton,Re-assemble,5.000000,5.0,5.0,12,5.0
1,Generic Talents,Race / Shalore,Timeless,4.916667,5.0,5.0,12,5.0
2,Generic Talents,Race / Ogre,Grisly Constitution,4.885714,5.0,5.0,35,5.0
3,Class Talents,Corruption / Wrath,Destroyer,4.869565,5.0,5.0,138,5.0
4,Generic Talents,Technique / Combat training,Weapons Mastery,4.728571,5.0,5.0,140,5.0
5,Generic Talents,Corruption / Demonic strength,Abyssal Shield,4.714286,5.0,5.0,140,5.0
6,Class Talents,Corruption / Brutality,Draining Assault,4.571429,5.0,5.0,140,5.0
7,Class Talents,Corruption / Torture,Abduction,4.371429,5.0,5.0,140,5.0
8,Class Talents,Corruption / Brutality,Reckless Strike,4.285714,5.0,5.0,140,5.0
9,Generic Talents,Corruption / Demonic strength,Demonic Blood,4.192857,5.0,5.0,140,5.0
